# DoubtLens AI — Reproducible NLP exploration

All bundled data are original synthetic examples. Scores do not represent real-world performance. Run from the repository root after installing `requirements.txt`. No outputs are fabricated.

## 1. Dataset provenance and license
Source: `scripts/generate_data.py`; CC0-1.0 original synthetic examples. Four categories, five shared templates per topic. Details in `docs/DATASET_CARD.md`.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from doubtlens.core import ROOT, prepare, split, baseline_model, alternate_model, measure, retrieve
import joblib

## 2. Inspect and clean the dataset

In [ ]:
raw = pd.read_csv(ROOT / "data/samples/synthetic_questions.csv")
print(raw.shape)
raw.head()

In [ ]:
data = prepare(raw)
print("After cleaning:", data.shape)
print("Exact duplicates removed:", len(raw)-len(data))
data.category.value_counts().plot.bar(title="Synthetic dataset class distribution")
plt.show()

## 3. Stratified split and leakage check
Vectorizers are trained only on training questions. Shared synthetic templates are present across splits: this is a remaining generalization problem.

In [ ]:
train, validation, test = split(data)
print(len(train), len(validation), len(test))
assert not set(train.question_text) & set(test.question_text)

## 4. Baseline: word TF-IDF + Logistic Regression

In [ ]:
base = baseline_model().fit(train.question_text, train.category)
print("Validation macro F1:", measure(base, validation)["macro_f1"])

## 5. Challenger: character TF-IDF + Logistic Regression

In [ ]:
char = alternate_model().fit(train.question_text, train.category)
print("Validation macro F1:", measure(char, validation)["macro_f1"])

## 6. Final held-out comparison
In deployed training, the model is selected by validation score. The held-out split is used only for final reporting.

In [ ]:
results = {"baseline": measure(base,test), "character": measure(char,test)}
pd.DataFrame({k:{m:v[m] for m in ["accuracy","macro_f1","weighted_f1"]} for k,v in results.items()}).T

## 7. Confusion matrix and error analysis

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from doubtlens.core import CATEGORIES
ConfusionMatrixDisplay(np.array(results["baseline"]["confusion_matrix"]),display_labels=CATEGORIES).plot(xticks_rotation=40)
plt.show()
pd.DataFrame(results["baseline"]["errors"])

## 8. Question retrieval and example predictions

In [ ]:
from doubtlens.core import predict
print(predict("Why does binary search need sorted data?",base))
index=joblib.load(ROOT / "artifacts/retrieval.joblib")
pd.DataFrame(retrieve("How should I debug React components?",index))

## 9. Optional pretrained sentence embeddings
Requires `pip install -r requirements-optional.txt` and external pretrained model weights. Run `python scripts/train_embeddings.py` from the repository root; its results are saved separately. Not executed as part of this notebook by default.

## 10. Limitations and next experiments
Templates induce near-duplicate patterns across splits. Replace synthetic questions with licensed human-labeled questions, group near-duplicates, evaluate on a fully independent test set, calibrate confidence estimates, and manually evaluate semantic retrieval quality.